In [ ]:
# PHASE 1D: portable runtime path bootstrap
import os as _tfm_os
from pathlib import Path as _TFMPath
_TFM_PUBLIC_PROJECT_ROOT = _TFMPath(
    _tfm_os.environ.get("TFM_PROJECT_ROOT", str(_TFMPath.cwd()))
).expanduser().resolve()
_TFM_PUBLIC_COLAB_WORKDIR = _TFMPath(
    _tfm_os.environ.get(
        "TFM_COLAB_WORKDIR",
        "/content" if _TFMPath("/content").is_dir() else str(_TFM_PUBLIC_PROJECT_ROOT),
    )
).expanduser().resolve()


# Phase E — TRANSFORMER-JUNE-BLIND-001 — ejecución completa v2

**Notebook:** `transformer_june_blind_full_colab_exec_v2_2026-08-22.ipynb`  
**Wrapper:** `TRANSFORMER-JUNE-BLIND-FULL-COLAB-EXEC-V2`  
**Fecha:** `2026-08-22`

## Objetivo

Ejecutar la evaluación externa ciega June del `TRANSFORMER*` ya congelado.

```text
TRANSFORMER* = T01 / L24
canonical seed = 20260820

H1
H3
H6
H12
```

Protocolo científico congelado:

```text
4 inner epoch-selection fits
+
4 final refits
=
8 training stages

expected blind predictions =
602 + 600 + 597 + 591
=
2390
```

## Reglas vinculantes

```text
best-seed selection = NO
retuning = NO
TRANSFORMER* change = NO
canonical seed change = NO
blind-target-driven fitting = NO
```

El wrapper valida `033–043`, el freeze, el bundle v2, June y el runtime antes del primer fit.


In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import subprocess
import sys
import time
import zipfile

import numpy as np
import pandas as pd
import torch

WRAPPER_ID = "TRANSFORMER-JUNE-BLIND-FULL-COLAB-EXEC-V2"
WRAPPER_VERSION = "2.0.0"
DATE_TAG = "2026-08-22"

EXPECTED_BUNDLE_NAME = "phase_e_transformer_june_blind_exec_v2_2026-08-22.zip"
EXPECTED_BUNDLE_SHA = "9f038b50b5f93722e1e8bda9c0708cbd09accb24f361c6126d178ffd29ef2a92"
EXPECTED_BUNDLE_ROOT = "phase_e_transformer_june_blind_exec_v2_2026-08-22"

EXPECTED_RUNNER_NAME = "run_transformer_june_blind_v2_2026-08-22.py"
EXPECTED_RUNNER_SHA = "fda523c8ecede8085e4fffc05dddef280308c8aa9e263bcc20c14091c2943fa1"

EXPECTED_BUNDLE_MANIFEST_NAME = (
    "phase_e_transformer_june_blind_exec_bundle_manifest_v2_2026-08-22.json"
)
EXPECTED_BUNDLE_MANIFEST_SHA = (
    "50348ece41073850acf57c7c622e9d6085a074aa70c789805d3abd53475d0b5c"
)

EXPECTED_043_NAME = "043_transformer_june_blind_preflight_closure_2026-08-22.md"
EXPECTED_043_SHA = "797c94fef0f6b5d43af61d3f5df4f712edf168a7edccf7694bd3124feb9cbf5d"

PREFLIGHT_REPORT_SHA = "c1aa84ff6f9959c0c9960c9303b7ff22cf137a62d999bb97205d9fa020c8624f"
PREFLIGHT_MANIFEST_SHA = "12d9a18008c691d8822063c8b3bccf7311f303c231cf16fec4fd40aca6e6ed72"

EXPECTED_FREEZE_MANIFEST_NAME = "transformer_pre_june_freeze_manifest_v1_2026-08-22.json"
EXPECTED_FREEZE_MANIFEST_SHA = "42f996fed2698e5f87e3feb9965e3155cc7331f86b6598f1b9f8fa4c2746182e"

EXPECTED_JUNE_NAME = "june_week3_prepared_5min.parquet"
EXPECTED_JUNE_SHA = "d528047b6d93d9b93f03b016c408c4a660447eb9c1ab1bd680f6dcbf628b56f3"

GOVERNANCE = {
    "033_phase_e_transformer_lite_protocol_2026-08-21.md":
        "4fcd61519612ae5fd3daa6d874a73df50b27b63531ac109f0782351875211552",
    "034_phase_e_colab_preflight_closure_2026-08-21.md":
        "9d89f0bbaa13448c56217c05f1663fa57fdb4505d3a395860186e2235e9cc081",
    "035_transformer_march_runner_preflight_closure_2026-08-21.md":
        "2f90c65e1855c98a20470d76229be68359cad8fb33b0b213c5f351eb9e21322e",
    "036_phase_e_colab_runtime_compatibility_2026-08-21.md":
        "8e8aeb3b0783c5928468bbbf074c1c42359837b57461890c4eb5e1145ca9dbf3",
    "037_transformer_march_screen_closure_2026-08-22.md":
        "9650e5aeb7ea8653e17baaa584237da8ab3d1fb41db9a150b0f98e29e075e03e",
    "038_transformer_april_selection_preflight_closure_2026-08-22.md":
        "99b9fbe0592729ec2c5e5d653124b62f52a423c917bda58568ab80ea67e5cc7c",
    "039_transformer_april_selection_closure_2026-08-22.md":
        "f4998ca12d138c91cc9d67f53b1e95a5bd4310cdb0637aa32e7690ec5836ab99",
    "040_transformer_stability_preflight_closure_2026-08-22.md":
        "31c1c4c3810746defb11b7f7bc26b99998cf51d432216e3dcda00c1413a525d9",
    "041_transformer_stability_closure_2026-08-22.md":
        "ffa68b530d479fe9d1510a7f74598c63b14a714c45316b86ad83454dd24d4c88",
    "042_transformer_pre_june_freeze_2026-08-22.md":
        "341050a9f41856792572764153ba75b1706b6ffad3252215d519b078f11531b0",
}

EXPECTED_RUNTIME = {
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "torch_cuda": "12.8",
    "cudnn": 91900,
    "numpy": "2.1.3",
    "pandas": "2.2.3",
}

EXPECTED_OUTPUT_ZIP = "ugr16_transformer_june_blind_outputs_v2_2026-08-22.zip"
EXPECTED_RUNNER_MANIFEST = "ugr16_transformer_june_blind_manifest_v2_2026-08-22.json"
EXPECTED_RUNTIME_CSV = "ugr16_transformer_june_blind_runtime_v2_2026-08-22.csv"
EXPECTED_EPOCH_CSV = "ugr16_transformer_june_blind_epoch_selection_v2_2026-08-22.csv"
EXPECTED_PREDICTIONS = "ugr16_transformer_june_blind_predictions_v2_2026-08-22.parquet"
EXPECTED_HORIZON_METRICS = "ugr16_transformer_june_blind_horizon_metrics_v2_2026-08-22.csv"
EXPECTED_RUNNER_REPORT = "ugr16_transformer_june_blind_report_v2_2026-08-22.txt"

CONSOLE_NAME = "transformer_june_blind_full_colab_exec_v2_2026-08-22_console.txt"
EXEC_MANIFEST_NAME = "transformer_june_blind_full_colab_exec_v2_2026-08-22_execution_manifest.json"

CONTENT = Path((str(_TFM_PUBLIC_COLAB_WORKDIR)))
UNPACK_PARENT = CONTENT / "transformer_june_blind_full_colab_exec_v2_unpack"
OUTPUT_DIR = CONTENT / "transformer_june_blind_full_colab_exec_v2_runner_outputs"

EXPECTED_RUNTIME_ROWS = 4
EXPECTED_EPOCH_ROWS = 4
EXPECTED_PREDICTION_ROWS = 2390
EXPECTED_HORIZON_ROWS = 4
EXPECTED_HORIZONS = [1, 3, 6, 12]
EXPECTED_COUNTS = {1: 602, 3: 600, 6: 597, 12: 591}

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def check(nombre, condicion, detalle=""):
    estado = "PASS" if bool(condicion) else "FAIL"
    linea = f"{nombre:76s}: {estado}" + (f" | {detalle}" if detalle else "")
    print(linea)
    if not condicion:
        raise RuntimeError(linea)

def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value

print("=" * 120)
print("INICIALIZACIÓN —", WRAPPER_ID)
print("=" * 120)
print("Bundle:", EXPECTED_BUNDLE_NAME)
print("Bundle SHA-256:", EXPECTED_BUNDLE_SHA)
print("Runner SHA-256:", EXPECTED_RUNNER_SHA)
print("043 SHA-256:", EXPECTED_043_SHA)
print("Expected training stages: 8")
print("Expected blind predictions: 2390")


## 1. Verificar el runtime autoritativo

El wrapper se detiene antes del upload si el runtime no coincide con `036`.


In [ ]:
runtime = {
    "python": sys.version.split()[0],
    "python_full": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_vram_bytes": (
        int(torch.cuda.get_device_properties(0).total_memory)
        if torch.cuda.is_available() else None
    ),
}

print(json.dumps(runtime, indent=2, ensure_ascii=False))

check("Python exacto según 036", runtime["python"] == EXPECTED_RUNTIME["python"], runtime["python"])
check("PyTorch exacto según 036", runtime["torch"] == EXPECTED_RUNTIME["torch"], runtime["torch"])
check("CUDA exacta según 036", runtime["torch_cuda"] == EXPECTED_RUNTIME["torch_cuda"], str(runtime["torch_cuda"]))
check("cuDNN exacto según 036", runtime["cudnn"] == EXPECTED_RUNTIME["cudnn"], str(runtime["cudnn"]))
check("NumPy exacto según 036", runtime["numpy"] == EXPECTED_RUNTIME["numpy"], runtime["numpy"])
check("pandas exacto según 036", runtime["pandas"] == EXPECTED_RUNTIME["pandas"], runtime["pandas"])
check("GPU CUDA disponible", runtime["cuda_available"], str(runtime["gpu_name"]))

print("STACK PHASE E 036: PASS")


## 2. Subir las dos autoridades de ejecución

Selecciona simultáneamente:

```text
phase_e_transformer_june_blind_exec_v2_2026-08-22.zip
043_transformer_june_blind_preflight_closure_2026-08-22.md
```

Los archivos se identifican por SHA-256, no por el nombre temporal que pueda asignar Colab.


In [ ]:
from google.colab import files

uploaded = files.upload()
nombres = sorted(uploaded.keys())

check("Se han subido exactamente dos ficheros", len(nombres) == 2, repr(nombres))

expected_by_sha = {
    EXPECTED_BUNDLE_SHA: {
        "canonical_name": EXPECTED_BUNDLE_NAME,
        "kind": "bundle",
    },
    EXPECTED_043_SHA: {
        "canonical_name": EXPECTED_043_NAME,
        "kind": "governance_043",
    },
}

received_by_kind = {}

for nombre_recibido, bytes_recibidos in uploaded.items():
    sha_recibido = hashlib.sha256(bytes_recibidos).hexdigest()

    print("Nombre recibido:", nombre_recibido)
    print("SHA-256 recibido:", sha_recibido)

    check(
        f"Upload reconocido por SHA: {nombre_recibido}",
        sha_recibido in expected_by_sha,
        sha_recibido,
    )

    spec = expected_by_sha[sha_recibido]
    kind = spec["kind"]
    check(f"Tipo {kind} no duplicado", kind not in received_by_kind)

    canonical = CONTENT / spec["canonical_name"]

    if canonical.exists():
        check(
            f"{kind}: canónico ya presente e idéntico",
            sha256(canonical) == sha_recibido,
            sha256(canonical),
        )
    else:
        canonical.write_bytes(bytes_recibidos)
        print("Creado basename canónico:", canonical.name)

    check(f"{kind}: SHA canónico exacto", sha256(canonical) == sha_recibido)

    temporal = CONTENT / nombre_recibido
    if nombre_recibido != canonical.name and temporal.exists():
        check(f"{kind}: copia temporal idéntica", sha256(temporal) == sha_recibido)
        temporal.unlink()

    received_by_kind[kind] = canonical

check("Bundle recibido", "bundle" in received_by_kind)
check("043 recibido", "governance_043" in received_by_kind)

bundle_zip = received_by_kind["bundle"]
closure043 = received_by_kind["governance_043"]

print("UPLOAD AUTORIZATIVO: PASS")


## 3. Gate previo al primer fit

Se audita completamente el bundle y se valida `043` externamente.

La ejecución científica no comienza hasta que esta celda termine en:

```text
GATE PREVIO AL ENTRENAMIENTO JUNE: PASS
```


In [ ]:
check("Directorio de extracción aún no existe", not UNPACK_PARENT.exists(), str(UNPACK_PARENT))
check("Directorio científico aún no existe", not OUTPUT_DIR.exists(), str(OUTPUT_DIR))

UNPACK_PARENT.mkdir(parents=True, exist_ok=False)

with zipfile.ZipFile(bundle_zip, "r") as zf:
    bad_member = zf.testzip()
    check("ZIP sin corrupción", bad_member is None, str(bad_member))
    zf.extractall(UNPACK_PARENT)

bundle_root = UNPACK_PARENT / EXPECTED_BUNDLE_ROOT
check("Raíz del bundle presente", bundle_root.is_dir(), str(bundle_root))

sha_file = bundle_root / "SHA256SUMS.txt"
check("SHA256SUMS.txt presente", sha_file.is_file(), str(sha_file))

proc_sha = subprocess.run(
    ["sha256sum", "-c", "SHA256SUMS.txt"],
    cwd=bundle_root,
    capture_output=True,
    text=True,
)

print(proc_sha.stdout, end="")
if proc_sha.stderr:
    print(proc_sha.stderr, file=sys.stderr, end="")

check("Validación SHA256SUMS interna", proc_sha.returncode == 0, str(proc_sha.returncode))

runner = bundle_root / "src" / EXPECTED_RUNNER_NAME
bundle_manifest = bundle_root / "manifest" / EXPECTED_BUNDLE_MANIFEST_NAME
freeze_manifest = bundle_root / "governance" / EXPECTED_FREEZE_MANIFEST_NAME
june = bundle_root / "inputs" / EXPECTED_JUNE_NAME

check("Runner v2 presente", runner.is_file(), str(runner))
check("Runner v2 SHA exacto", sha256(runner) == EXPECTED_RUNNER_SHA, sha256(runner))

check("Manifest PRE-EXECUTION presente", bundle_manifest.is_file(), str(bundle_manifest))
check(
    "Manifest PRE-EXECUTION SHA exacto",
    sha256(bundle_manifest) == EXPECTED_BUNDLE_MANIFEST_SHA,
    sha256(bundle_manifest),
)

for name, expected_sha in GOVERNANCE.items():
    p = bundle_root / "governance" / name
    check(f"{name}: presente", p.is_file(), str(p))
    check(f"{name}: SHA exacto", sha256(p) == expected_sha, sha256(p))

check("Freeze manifest presente", freeze_manifest.is_file(), str(freeze_manifest))
check(
    "Freeze manifest SHA exacto",
    sha256(freeze_manifest) == EXPECTED_FREEZE_MANIFEST_SHA,
    sha256(freeze_manifest),
)

check("043 externo presente", closure043.is_file(), str(closure043))
check("043 externo SHA exacto", sha256(closure043) == EXPECTED_043_SHA, sha256(closure043))

input_files = sorted(
    p.name for p in (bundle_root / "inputs").iterdir() if p.is_file()
)
check(
    "inputs/ contiene exactamente June",
    input_files == [EXPECTED_JUNE_NAME],
    repr(input_files),
)

check("June presente", june.is_file(), str(june))
check("June SHA exacto", sha256(june) == EXPECTED_JUNE_SHA, sha256(june))

meta = json.loads(bundle_manifest.read_text(encoding="utf-8"))
check("Campaign correcta", meta.get("campaign_id") == "TRANSFORMER-JUNE-BLIND-001")
check("Estado PRE-EXECUTION", meta.get("status") == "PRE-EXECUTION / FROZEN_INPUT_BUNDLE")

cfg = meta.get("frozen_configuration", {})
check("TRANSFORMER* = T01", cfg.get("config_id") == "T01", repr(cfg))
check("Lookback = 24", cfg.get("lookback") == 24, repr(cfg))
check("Seed canónica = 20260820", cfg.get("canonical_seed") == 20260820, repr(cfg))
check("Parámetros = 8705", cfg.get("trainable_parameters") == 8705, repr(cfg))

expected_exec = meta.get("expected_execution", {})
check("Expected inner fits = 4", expected_exec.get("inner_epoch_selection_fits") == 4)
check("Expected final refits = 4", expected_exec.get("final_refits") == 4)
check("Expected stages = 8", expected_exec.get("training_stages") == 8)
check("Expected blind predictions = 2390", expected_exec.get("blind_predictions") == 2390)
check("Expected horizon rows = 4", expected_exec.get("horizon_metric_rows") == 4)

pre = meta.get("pre_execution_assertions", {})
check("No training previo", pre.get("training_executed") is False)
check("No blind predictions previas", pre.get("blind_predictions_generated") is False)
check("Selección prohibida", pre.get("selection_allowed") is False)
check("Retuning prohibido", pre.get("retuning_allowed") is False)
check("Cambio TRANSFORMER* prohibido", pre.get("transformer_star_change_allowed") is False)
check("Cambio seed canónica prohibido", pre.get("canonical_seed_change_allowed") is False)
check("Blind-target fitting prohibido", pre.get("blind_target_driven_fitting_allowed") is False)

print("GATE PREVIO AL ENTRENAMIENTO JUNE: PASS")


## 4. Ejecutar exactamente las 8 etapas científicas

A partir de esta celda se ejecuta:

```text
H1: inner + refit
H3: inner + refit
H6: inner + refit
H12: inner + refit
```

Total:

```text
8 training stages
2390 blind predictions expected
```


In [ ]:
check("Directorio de outputs aún ausente", not OUTPUT_DIR.exists(), str(OUTPUT_DIR))

cmd = [
    sys.executable,
    str(runner),
    "--bundle-root",
    str(bundle_root),
    "--output-dir",
    str(OUTPUT_DIR),
]

print("Comando:")
print(" ".join(cmd))
print()

started = time.perf_counter()

proc = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
)

wrapper_elapsed = time.perf_counter() - started

console_path = CONTENT / CONSOLE_NAME
check("Consola v2 aún no existe", not console_path.exists(), str(console_path))

console_text = proc.stdout
if proc.stderr:
    console_text += "\n[STDERR]\n" + proc.stderr

console_path.write_text(console_text, encoding="utf-8")

print(proc.stdout)
if proc.stderr:
    print("\n[STDERR]")
    print(proc.stderr)

check("Código de retorno del runner = 0", proc.returncode == 0, str(proc.returncode))

required_stdout = [
    "TRANSFORMER-JUNE-BLIND-001: PASS",
    "TRANSFORMER* CHANGED: NO",
    "BEST SEED SELECTED: NO",
    "RETUNING: NO",
    "TRANSFORMER-INFERENCE-RESIDUALS-001: NOT RUN",
    "TRANSFORMER-CLOSURE-001: NOT RUN",
]

for phrase in required_stdout:
    check(f"stdout contiene: {phrase}", phrase in proc.stdout)

print("Tiempo total del wrapper (s):", wrapper_elapsed)
print("EJECUCIÓN JUNE DEL RUNNER: PASS")


## 5. Auditar los outputs científicos June

Deben existir exactamente los outputs científicos v2 del runner y cumplir:

```text
runtime rows = 4
epoch-selection rows = 4
blind prediction rows = 2390
horizon metric rows = 4
```


In [ ]:
runtime_csv = OUTPUT_DIR / EXPECTED_RUNTIME_CSV
epoch_csv = OUTPUT_DIR / EXPECTED_EPOCH_CSV
predictions_path = OUTPUT_DIR / EXPECTED_PREDICTIONS
horizon_metrics_path = OUTPUT_DIR / EXPECTED_HORIZON_METRICS
runner_report = OUTPUT_DIR / EXPECTED_RUNNER_REPORT
runner_manifest = OUTPUT_DIR / EXPECTED_RUNNER_MANIFEST
runner_zip = OUTPUT_DIR / EXPECTED_OUTPUT_ZIP

for label, p in [
    ("Runtime CSV", runtime_csv),
    ("Epoch-selection CSV", epoch_csv),
    ("Predictions parquet", predictions_path),
    ("Horizon metrics CSV", horizon_metrics_path),
    ("Runner report", runner_report),
    ("Runner manifest", runner_manifest),
    ("Results ZIP", runner_zip),
]:
    check(f"{label} presente", p.is_file(), str(p))

runtime_df = pd.read_csv(runtime_csv)
epoch_df = pd.read_csv(epoch_csv)
pred_df = pd.read_parquet(predictions_path)
metrics_df = pd.read_csv(horizon_metrics_path)
runner_meta = json.loads(runner_manifest.read_text(encoding="utf-8"))

check("Manifest científico status PASS", runner_meta.get("status") == "PASS", str(runner_meta.get("status")))
check("Manifest campaign id correcto", runner_meta.get("campaign_id") == "TRANSFORMER-JUNE-BLIND-001")

check("Runtime rows = 4", len(runtime_df) == EXPECTED_RUNTIME_ROWS, str(len(runtime_df)))
check("Epoch-selection rows = 4", len(epoch_df) == EXPECTED_EPOCH_ROWS, str(len(epoch_df)))
check("Predictions rows = 2390", len(pred_df) == EXPECTED_PREDICTION_ROWS, str(len(pred_df)))
check("Horizon metrics rows = 4", len(metrics_df) == EXPECTED_HORIZON_ROWS, str(len(metrics_df)))

check("Todos los runtime status PASS", bool(runtime_df["status"].eq("PASS").all()))

check(
    "Horizontes runtime exactos",
    sorted(runtime_df["horizon_steps"].astype(int).tolist()) == EXPECTED_HORIZONS,
    repr(sorted(runtime_df["horizon_steps"].astype(int).tolist())),
)
check(
    "Horizontes epoch-selection exactos",
    sorted(epoch_df["horizon_steps"].astype(int).tolist()) == EXPECTED_HORIZONS,
)
check(
    "Horizontes metrics exactos",
    sorted(metrics_df["horizon_steps"].astype(int).tolist()) == EXPECTED_HORIZONS,
)

for h, expected_count in EXPECTED_COUNTS.items():
    n = int(pred_df["horizon_steps"].astype(int).eq(h).sum())
    check(f"H{h} prediction count", n == expected_count, str(n))

check(
    "Predicciones finitas",
    bool(np.isfinite(pred_df["y_pred_bps"].to_numpy(float)).all()),
)

check(
    "TRANSFORMER* config runtime = T01",
    bool(runtime_df["config_id"].astype(str).eq("T01").all()),
)
check(
    "Lookback runtime = 24",
    bool(runtime_df["lookback"].astype(int).eq(24).all()),
)
check(
    "Seed canónica runtime = 20260820",
    bool(runtime_df["canonical_seed"].astype(int).eq(20260820).all()),
)
check(
    "Parámetros runtime = 8705",
    bool(runtime_df["parameter_count"].astype(int).eq(8705).all()),
)

campaign = runner_meta.get("campaign", {})
check("4 inner fits manifest", campaign.get("inner_epoch_selection_fits") == 4, repr(campaign))
check("4 final refits manifest", campaign.get("final_refits") == 4, repr(campaign))
check("8 total stages manifest", campaign.get("total_training_stages") == 8, repr(campaign))
check("2390 blind rows manifest", campaign.get("blind_prediction_rows") == 2390, repr(campaign))
check("TRANSFORMER* unchanged manifest", campaign.get("transformer_star_changed") is False)
check("Best-seed selection false manifest", campaign.get("best_seed_selection") is False)
check("Retuning false manifest", campaign.get("retuning") is False)
check("Blind-target fitting false manifest", campaign.get("blind_targets_used_for_fitting") is False)

validation = runner_meta.get("validation", {})
check("Validaciones científicas presentes", bool(validation), repr(validation))
check("Todas las validaciones científicas PASS", all(bool(v) for v in validation.values()), repr(validation))

check("Bundle original inmutable", sha256(bundle_zip) == EXPECTED_BUNDLE_SHA, sha256(bundle_zip))
check("Runner original inmutable", sha256(runner) == EXPECTED_RUNNER_SHA, sha256(runner))
check(
    "Manifest PRE-EXECUTION inmutable",
    sha256(bundle_manifest) == EXPECTED_BUNDLE_MANIFEST_SHA,
    sha256(bundle_manifest),
)
check("043 externo inmutable", sha256(closure043) == EXPECTED_043_SHA, sha256(closure043))
check("June inmutable", sha256(june) == EXPECTED_JUNE_SHA, sha256(june))

print("\nMÉTRICAS BLIND JUNE")
cols = [
    "horizon_steps",
    "n_predictions",
    "best_epoch",
    "mae_bps",
    "rmse_bps",
    "mae_ratio_vs_persistence",
    "skill_vs_persistence",
    "bias_bps",
    "underprediction_pct",
    "p95_abs_error_bps",
]
print(metrics_df[cols].sort_values("horizon_steps").to_string(index=False))

print("AUDITORÍA DE OUTPUTS JUNE: PASS")


## 6. Verificar el ZIP científico

El ZIP del runner debe contener exactamente seis outputs científicos:

```text
runtime CSV
epoch-selection CSV
predictions parquet
horizon metrics CSV
report TXT
manifest JSON
```


In [ ]:
with zipfile.ZipFile(runner_zip, "r") as zf:
    bad_member = zf.testzip()
    check("ZIP científico sin corrupción", bad_member is None, str(bad_member))
    members = sorted(zf.namelist())

expected_members = sorted([
    EXPECTED_RUNTIME_CSV,
    EXPECTED_EPOCH_CSV,
    EXPECTED_PREDICTIONS,
    EXPECTED_HORIZON_METRICS,
    EXPECTED_RUNNER_REPORT,
    EXPECTED_RUNNER_MANIFEST,
])

print("Contenido del ZIP científico:")
for name in members:
    print(" -", name)

check(
    "ZIP contiene exactamente los seis outputs",
    members == expected_members,
    repr(members),
)

print("SHA-256 ZIP científico:", sha256(runner_zip))
print("ZIP CIENTÍFICO JUNE: PASS")


## 7. Crear el manifest de ejecución Colab

Este manifest del wrapper preserva explícitamente:

```text
TRANSFORMER* unchanged
best-seed selection = NO
retuning = NO
blind-target-driven fitting = NO
```

También registra los hashes de los tres artefactos que se devolverán.


In [ ]:
metric_records = []
for row in metrics_df.sort_values("horizon_steps").itertuples(index=False):
    metric_records.append({
        "horizon_steps": int(row.horizon_steps),
        "n_predictions": int(row.n_predictions),
        "best_epoch": int(row.best_epoch),
        "mae_bps": float(row.mae_bps),
        "rmse_bps": float(row.rmse_bps),
        "mae_ratio_vs_persistence": float(row.mae_ratio_vs_persistence),
        "skill_vs_persistence": float(row.skill_vs_persistence),
        "bias_bps": float(row.bias_bps),
        "underprediction_pct": float(row.underprediction_pct),
        "p95_abs_error_bps": float(row.p95_abs_error_bps),
    })

exec_manifest = {
    "wrapper_id": WRAPPER_ID,
    "wrapper_version": WRAPPER_VERSION,
    "date_tag": DATE_TAG,
    "status": "PASS",
    "bundle": {
        "name": EXPECTED_BUNDLE_NAME,
        "sha256": sha256(bundle_zip),
        "expected_sha256": EXPECTED_BUNDLE_SHA,
    },
    "bundle_manifest": {
        "name": EXPECTED_BUNDLE_MANIFEST_NAME,
        "sha256": sha256(bundle_manifest),
        "expected_sha256": EXPECTED_BUNDLE_MANIFEST_SHA,
    },
    "runner": {
        "name": EXPECTED_RUNNER_NAME,
        "sha256": sha256(runner),
        "expected_sha256": EXPECTED_RUNNER_SHA,
        "return_code": int(proc.returncode),
    },
    "governance": {
        "closure_043_name": EXPECTED_043_NAME,
        "closure_043_sha256": sha256(closure043),
        "preflight_report_sha256": PREFLIGHT_REPORT_SHA,
        "preflight_manifest_sha256": PREFLIGHT_MANIFEST_SHA,
        "freeze_manifest_sha256": EXPECTED_FREEZE_MANIFEST_SHA,
    },
    "june": {
        "name": EXPECTED_JUNE_NAME,
        "sha256": sha256(june),
        "role": "final blind external evaluation",
    },
    "runtime": runtime,
    "campaign": {
        "id": "TRANSFORMER-JUNE-BLIND-001",
        "status": runner_meta.get("status"),
        "transformer_star": {"config_id": "T01", "lookback": 24},
        "canonical_seed": 20260820,
        "inner_epoch_selection_fits": 4,
        "final_refits": 4,
        "total_training_stages": 8,
        "blind_prediction_rows": int(len(pred_df)),
        "horizon_metric_rows": int(len(metrics_df)),
        "transformer_star_changed": False,
        "best_seed_selection": False,
        "retuning": False,
        "blind_targets_used_for_fitting": False,
    },
    "blind_metrics": metric_records,
    "artifacts": {
        "console": {
            "name": console_path.name,
            "sha256": sha256(console_path),
            "bytes": int(console_path.stat().st_size),
        },
        "results_zip": {
            "name": runner_zip.name,
            "sha256": sha256(runner_zip),
            "bytes": int(runner_zip.stat().st_size),
        },
        "runner_manifest": {
            "name": runner_manifest.name,
            "sha256": sha256(runner_manifest),
            "bytes": int(runner_manifest.stat().st_size),
        },
    },
    "checks": {
        "bundle_sha_pass": sha256(bundle_zip) == EXPECTED_BUNDLE_SHA,
        "internal_bundle_integrity_pass": proc_sha.returncode == 0,
        "runner_return_code_zero": proc.returncode == 0,
        "runner_campaign_pass": runner_meta.get("status") == "PASS",
        "runtime_rows_4": len(runtime_df) == 4,
        "epoch_selection_rows_4": len(epoch_df) == 4,
        "blind_prediction_rows_2390": len(pred_df) == 2390,
        "horizon_metric_rows_4": len(metrics_df) == 4,
        "runner_validations_all_pass": bool(validation) and all(bool(v) for v in validation.values()),
        "transformer_star_unchanged": campaign.get("transformer_star_changed") is False,
        "best_seed_selection_false": campaign.get("best_seed_selection") is False,
        "retuning_false": campaign.get("retuning") is False,
        "blind_target_fitting_false": campaign.get("blind_targets_used_for_fitting") is False,
        "bundle_immutable": sha256(bundle_zip) == EXPECTED_BUNDLE_SHA,
        "runner_immutable": sha256(runner) == EXPECTED_RUNNER_SHA,
        "bundle_manifest_immutable": sha256(bundle_manifest) == EXPECTED_BUNDLE_MANIFEST_SHA,
        "closure_043_immutable": sha256(closure043) == EXPECTED_043_SHA,
        "june_immutable": sha256(june) == EXPECTED_JUNE_SHA,
    },
    "wrapper_elapsed_seconds": float(wrapper_elapsed),
}

check(
    "Todos los checks del execution manifest PASS",
    all(bool(v) for v in exec_manifest["checks"].values()),
    repr(exec_manifest["checks"]),
)

exec_manifest_path = CONTENT / EXEC_MANIFEST_NAME
check("Execution manifest v2 aún no existe", not exec_manifest_path.exists(), str(exec_manifest_path))

exec_manifest_path.write_text(
    json.dumps(
        json_safe(exec_manifest),
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    ) + "\n",
    encoding="utf-8",
)

print("Execution manifest:", exec_manifest_path)
print("SHA-256 execution manifest:", sha256(exec_manifest_path))
print("MANIFEST DE EJECUCIÓN COLAB: PASS")


## 8. Resumen final

Si esta celda termina correctamente, June habrá sido ejecutado una única vez conforme al freeze.

**No ejecutes todavía `TRANSFORMER-INFERENCE-RESIDUALS-001` ni Phase F.**


In [ ]:
print("=" * 120)
print("RESUMEN FINAL — TRANSFORMER-JUNE-BLIND-001")
print("=" * 120)

print("STATUS: PASS")
print("TRANSFORMER*: T01 / L24")
print("CANONICAL SEED: 20260820")
print("TRAINING STAGES:", 8)
print("BLIND PREDICTIONS:", len(pred_df))
print("HORIZON METRIC ROWS:", len(metrics_df))
print()

for row in metric_records:
    print(
        f"H{row['horizon_steps']:<2d} "
        f"n={row['n_predictions']} "
        f"best_epoch={row['best_epoch']} "
        f"MAE={row['mae_bps']:.6f} "
        f"ratio={row['mae_ratio_vs_persistence']:.9f} "
        f"skill={row['skill_vs_persistence']:.9f} "
        f"bias={row['bias_bps']:.6f}"
    )

print()
print("TRANSFORMER* CHANGED: NO")
print("BEST SEED SELECTED: NO")
print("RETUNING: NO")
print("BLIND-TARGET-DRIVEN FITTING: NO")
print()
print("SHA ZIP resultados:", sha256(runner_zip))
print("SHA consola:", sha256(console_path))
print("SHA manifest runner:", sha256(runner_manifest))
print("SHA manifest ejecución:", sha256(exec_manifest_path))
print()
print("TRANSFORMER-JUNE-BLIND-001: PASS")
print("TRANSFORMER-INFERENCE-RESIDUALS-001: NOT RUN")
print("TRANSFORMER-CLOSURE-001: NOT RUN")
print("PHASE F: NOT STARTED")
print("=" * 120)


## 9. Descargar los tres artefactos de retorno

Descarga exactamente:

1. ZIP científico June.
2. Consola completa.
3. Manifest de ejecución Colab.

Adjunta los tres al chat técnico.

**No ejecutes todavía el análisis de inferencia/residuos ni Phase F.**


In [ ]:
from google.colab import files

files.download(str(runner_zip))
files.download(str(console_path))
files.download(str(exec_manifest_path))

print("=" * 120)
print("DESCARGA SOLICITADA")
print("=" * 120)
print("1.", runner_zip.name)
print("2.", console_path.name)
print("3.", exec_manifest_path.name)
print()
print("NO EJECUTAR TRANSFORMER-INFERENCE-RESIDUALS-001")
print("NO INICIAR PHASE F")
print("=" * 120)
